In [62]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [63]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document


#vector store
from langchain_chroma import Chroma

##utility imports
import numpy as np
from typing import List


In [64]:
sample_text = [
    """\n\nMachine Learning\n\n
Machine learning is a branch of artificial intelligence in which systems learn patterns from data instead of being programmed with explicit rules for every case.
Algorithms build mathematical models from examples—labeled data in supervised learning, unlabeled data in unsupervised learning, or rewards and penalties in reinforcement learning—and use those models to make predictions or decisions on new inputs.
Common applications include spam filtering, recommendation systems, fraud detection, and medical diagnosis.
The field rests on statistics, optimization, and computer science, and its strength is generalization: a well-trained model can perform well on data it has never seen, as long as that data resembles what it was trained on.
""",

    """\n\nDeep Learning\n\n
Deep learning is a subset of machine learning built on artificial neural networks with many layers, which is why it is called “deep.”
These networks learn hierarchical representations: early layers may detect simple features such as edges or words, while deeper layers combine them into richer concepts such as objects, sentences, or abstract patterns.
Deep learning has driven major advances in image recognition, speech processing, game playing, and generative AI, largely because it scales well with large datasets and powerful hardware such as GPUs.
Models like convolutional neural networks (CNNs) excel at spatial data like images, while recurrent and transformer-based architectures are widely used for sequential and language data.
""",

    """\n\nNatural Language Processing (NLP)\n\n
Natural language processing (NLP) focuses on enabling computers to understand, interpret, and generate human language.
It spans tasks such as text classification, sentiment analysis, machine translation, question answering, summarization, and conversational AI.
Traditional NLP relied heavily on linguistic rules and handcrafted features, but modern NLP is dominated by deep learning—especially transformer models like BERT and GPT—which learn context and meaning from vast amounts of text.
NLP powers virtual assistants, search engines, chatbots, and document analysis systems, and it sits at the center of today’s large language models, which can read, write, and reason over language with remarkable fluency.
"""
]

sample_text

['\n\nMachine Learning\n\n\nMachine learning is a branch of artificial intelligence in which systems learn patterns from data instead of being programmed with explicit rules for every case.\nAlgorithms build mathematical models from examples—labeled data in supervised learning, unlabeled data in unsupervised learning, or rewards and penalties in reinforcement learning—and use those models to make predictions or decisions on new inputs.\nCommon applications include spam filtering, recommendation systems, fraud detection, and medical diagnosis.\nThe field rests on statistics, optimization, and computer science, and its strength is generalization: a well-trained model can perform well on data it has never seen, as long as that data resembles what it was trained on.\n',
 '\n\nDeep Learning\n\n\nDeep learning is a subset of machine learning built on artificial neural networks with many layers, which is why it is called “deep.”\nThese networks learn hierarchical representations: early layers

In [65]:
import tempfile
temp_dir = tempfile.mkdtemp()

for i,doc in enumerate(sample_text):
    with open(f"doc_{i}.txt", "w") as f:
        f.write(doc)



In [66]:
from langchain_community.document_loaders import DirectoryLoader

loader = DirectoryLoader(
    "data",
    glob = "*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=True,
)

docs = loader.load()


print(docs)

100%|██████████| 3/3 [00:00<00:00, 247.38it/s]

[Document(metadata={'source': 'data/doc_2.txt'}, page_content='\n\nNatural Language Processing (NLP)\n\n\nNatural language processing (NLP) focuses on enabling computers to understand, interpret, and generate human language.\nIt spans tasks such as text classification, sentiment analysis, machine translation, question answering, summarization, and conversational AI.\nTraditional NLP relied heavily on linguistic rules and handcrafted features, but modern NLP is dominated by deep learning—especially transformer models like BERT and GPT—which learn context and meaning from vast amounts of text.\nNLP powers virtual assistants, search engines, chatbots, and document analysis systems, and it sits at the center of today’s large language models, which can read, write, and reason over language with remarkable fluency.\n'), Document(metadata={'source': 'data/doc_0.txt'}, page_content='\n\nMachine Learning\n\n\nMachine learning is a branch of artificial intelligence in which systems learn pattern

In [67]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=[" "]
)

texts = text_splitter.split_documents(docs)


In [68]:
texts

[Document(metadata={'source': 'data/doc_2.txt'}, page_content='Natural Language Processing (NLP)\n\n\nNatural language processing (NLP) focuses on enabling computers to understand, interpret, and generate human language.\nIt spans tasks such as text classification, sentiment analysis, machine translation, question answering, summarization, and conversational AI.\nTraditional NLP relied heavily on linguistic rules and handcrafted features, but modern NLP is dominated by deep learning—especially transformer models like BERT and GPT—which learn context and'),
 Document(metadata={'source': 'data/doc_2.txt'}, page_content='models like BERT and GPT—which learn context and meaning from vast amounts of text.\nNLP powers virtual assistants, search engines, chatbots, and document analysis systems, and it sits at the center of today’s large language models, which can read, write, and reason over language with remarkable fluency.'),
 Document(metadata={'source': 'data/doc_0.txt'}, page_content='Ma

In [69]:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [70]:
sample_text = "Machine Learning is Fascinating"
embedding = OpenAIEmbeddings()
embedding

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x13745e810>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x1377a7740>, model='text-embedding-ada-002', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [71]:
vector = embedding.embed_query(sample_text)
vector


[-0.02159224823117256,
 0.021384380757808685,
 0.013251559808850288,
 -0.02277449518442154,
 -0.005300623830407858,
 0.020539917051792145,
 -0.0049823266454041,
 0.00708698621019721,
 -0.016005804762244225,
 -0.043054576963186264,
 0.012348635122179985,
 0.0372602678835392,
 -0.017266003414988518,
 -0.0019714944064617157,
 -0.004670525435358286,
 0.006262011826038361,
 0.03728625178337097,
 0.011822470463812351,
 -0.0013170361053198576,
 -0.003621443407610059,
 -0.02795819379389286,
 0.03167382627725601,
 -0.004888136871159077,
 -0.03801378980278969,
 -0.0238787904381752,
 0.0029133944772183895,
 -0.0002450158353894949,
 -0.04261285811662674,
 -0.011465197429060936,
 -0.004322996828705072,
 0.026022426784038544,
 -0.003501269966363907,
 -0.020448975265026093,
 -0.03627289831638336,
 -0.00841863825917244,
 -0.01050380989909172,
 -0.0006296926876530051,
 -0.004586079157888889,
 -0.017071127891540527,
 0.0006162949721328914,
 0.024645302444696426,
 0.024294527247548103,
 -0.00387153425253

In [72]:
## Create a Chroma vector store
persist_directory = "./chroma_db"

vectors_store = Chroma.from_documents(
    documents=texts,
    embedding=embedding,
    persist_directory=persist_directory,
    collection_name="rag_collection",
)

print(vectors_store._collection.count())
print("persist directory: ", persist_directory)

22
persist directory:  ./chroma_db


In [73]:
query = "What is Machine Learning?"
result = vectors_store.similarity_search(query,k=3)
result

[Document(id='c83a6ed1-fcfb-46e3-b6b9-111b2a00e7c0', metadata={'source': 'data/doc_0.txt'}, page_content='Machine Learning\n\n\nMachine learning is a branch of artificial intelligence in which systems learn patterns from data instead of being programmed with explicit rules for every case.\nAlgorithms build mathematical models from examples—labeled data in supervised learning, unlabeled data in unsupervised learning, or rewards and penalties in reinforcement learning—and use those models to make predictions or decisions on new inputs.\nCommon applications include spam filtering, recommendation systems,'),
 Document(id='fba32b6a-a98c-455c-8ba7-cd4dd23a8a27', metadata={'source': 'data/doc_0.txt'}, page_content='Machine Learning\n\n\nMachine learning is a branch of artificial intelligence in which systems learn patterns from data instead of being programmed with explicit rules for every case.\nAlgorithms build mathematical models from examples—labeled data in supervised learning, unlabeled

In [74]:
result_with_score = vectors_store.similarity_search_with_score(query,k=3)
result_with_score

[(Document(id='c83a6ed1-fcfb-46e3-b6b9-111b2a00e7c0', metadata={'source': 'data/doc_0.txt'}, page_content='Machine Learning\n\n\nMachine learning is a branch of artificial intelligence in which systems learn patterns from data instead of being programmed with explicit rules for every case.\nAlgorithms build mathematical models from examples—labeled data in supervised learning, unlabeled data in unsupervised learning, or rewards and penalties in reinforcement learning—and use those models to make predictions or decisions on new inputs.\nCommon applications include spam filtering, recommendation systems,'),
  0.19662684202194214),
 (Document(id='fba32b6a-a98c-455c-8ba7-cd4dd23a8a27', metadata={'source': 'data/doc_0.txt'}, page_content='Machine Learning\n\n\nMachine learning is a branch of artificial intelligence in which systems learn patterns from data instead of being programmed with explicit rules for every case.\nAlgorithms build mathematical models from examples—labeled data in supe

In [75]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-3.5-turbo")


In [76]:
test_response = llm.invoke("What is Machine Learning?")
test_response

AIMessage(content='Machine learning is a subset of artificial intelligence that focuses on the development of algorithms and statistical models that allow computers to improve their performance on a specific task through experience or data without being explicitly programmed. In other words, machine learning enables machines to learn from and make predictions or decisions based on data. This technology is widely used in various fields such as image recognition, natural language processing, and data analysis.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 79, 'prompt_tokens': 12, 'total_tokens': 91, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DyegG5N8hV8IZTq7tuKFJ3vzaIWlP', 

In [77]:
from langchain.chat_models.base import init_chat_model

llm = init_chat_model(model="gpt-3.5-turbo")
llm

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11', 'langchain-openai': '1.3.3'}}, output_version=None, profile={'name': 'GPT-3.5-turbo', 'release_date': '2023-03-01', 'last_updated': '2023-11-06', 'open_weights': False, 'max_input_tokens': 16385, 'max_output_tokens': 4096, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': False, 'structured_output': False, 'attachment': False, 'temperature': True, 'image_url_inputs': False, 'pdf_inputs': False, 'pdf_tool_message': False, 'image_tool_message': False, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x1377957f0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x137569d30>, root_client=<openai.OpenAI object at 0x1

In [78]:
test_response = llm.invoke("What is Machine Learning?")
test_response

AIMessage(content='Machine learning is a subset of artificial intelligence that involves the development of algorithms and statistical models that computer systems use to learn from and make predictions or decisions based on data without being explicitly programmed. It enables computers to learn and improve from experience, without human intervention. Machine learning encompasses various techniques such as supervised learning, unsupervised learning, and reinforcement learning that enable computers to recognize patterns, make predictions, and solve problems.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 83, 'prompt_tokens': 12, 'total_tokens': 95, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None

In [79]:
## Modern RAG CHAIN 

from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate



In [80]:
retriever = vectors_store.as_retriever(
    search_kwargs={"k": 3}
)
retriever

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x13778c410>, search_kwargs={'k': 3})

In [81]:
system_prompt = """
You are a helpful assistant that can answer questions about the provided text. Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know. Don't make up an answer.
Use three maximum sentences to answer the question.
Context: {context}
"""

prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])




In [82]:
prompt_template

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="\nYou are a helpful assistant that can answer questions about the provided text. Use the following pieces of retrieved context to answer the question.\nIf you don't know the answer, just say that you don't know. Don't make up an answer.\nUse three maximum sentences to answer the question.\nContext: {context}\n"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

In [83]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
document_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=prompt_template,
)
document_chain



RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="\nYou are a helpful assistant that can answer questions about the provided text. Use the following pieces of retrieved context to answer the question.\nIf you don't know the answer, just say that you don't know. Don't make up an answer.\nUse three maximum sentences to answer the question.\nContext: {context}\n"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11',

In [84]:
rag_chain = create_retrieval_chain(retriever,document_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x13778c410>, search_kwargs={'k': 3}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="\nYou are a helpful assistant that can answer questions about the provided text. Use the following pieces of retrieved context to answer the questi

In [85]:
response = rag_chain.invoke({"input": "What is Machine Learning?"})

In [86]:
response["answer"]

'Machine learning is a branch of artificial intelligence where systems learn patterns from data rather than using explicit rules for all scenarios. Algorithms create mathematical models from labeled, unlabeled data in supervised and unsupervised learning, or rewards and penalties in reinforcement learning, to make predictions or decisions on new data inputs. Its common applications include spam filtering and recommendation systems.'

## Create RAG Chain Alternative - Using LCEL (Langchain Expression Language)

In [87]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel 

In [88]:
# Create a prompt 

custom_prompt = ChatPromptTemplate.from_template(
    """
    Use the following pieces of retrieved context to answer the question.
    If you don't know the answer, just say that you don't know. Don't make up an answer.
    Provide specific details from the context to support the answer.
    Context: {context}
    Question: {question}
    Answer:"""
)
custom_prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="\n    Use the following pieces of retrieved context to answer the question.\n    If you don't know the answer, just say that you don't know. Don't make up an answer.\n    Provide specific details from the context to support the answer.\n    Context: {context}\n    Question: {question}\n    Answer:"), additional_kwargs={})])

In [89]:
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

In [90]:
rag_chain_lcel = (
    {"context":retriever | format_docs, "question":RunnablePassthrough()}
    | custom_prompt
    | llm
    | StrOutputParser()
)
rag_chain_lcel


{
  context: VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x13778c410>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="\n    Use the following pieces of retrieved context to answer the question.\n    If you don't know the answer, just say that you don't know. Don't make up an answer.\n    Provide specific details from the context to support the answer.\n    Context: {context}\n    Question: {question}\n    Answer:"), additional_kwargs={})])
| ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11', 'langchain-openai': '1.3.3'}}, output_version=None, profile={'name': 'GPT-3.5-turbo', 'rel

In [91]:
response = rag_chain_lcel.invoke("What is Machine Learning?")
response

'Machine Learning is a branch of artificial intelligence where systems learn patterns from data rather than being explicitly programmed with rules for each case. Algorithms create mathematical models using labeled data in supervised learning, unlabeled data in unsupervised learning, or rewards and penalties in reinforcement learning to make predictions or decisions on new inputs. Common applications of machine learning include spam filtering and recommendation systems.'

In [92]:
retriever.invoke("What is Machine Learning?")

[Document(id='c83a6ed1-fcfb-46e3-b6b9-111b2a00e7c0', metadata={'source': 'data/doc_0.txt'}, page_content='Machine Learning\n\n\nMachine learning is a branch of artificial intelligence in which systems learn patterns from data instead of being programmed with explicit rules for every case.\nAlgorithms build mathematical models from examples—labeled data in supervised learning, unlabeled data in unsupervised learning, or rewards and penalties in reinforcement learning—and use those models to make predictions or decisions on new inputs.\nCommon applications include spam filtering, recommendation systems,'),
 Document(id='fba32b6a-a98c-455c-8ba7-cd4dd23a8a27', metadata={'source': 'data/doc_0.txt'}, page_content='Machine Learning\n\n\nMachine learning is a branch of artificial intelligence in which systems learn patterns from data instead of being programmed with explicit rules for every case.\nAlgorithms build mathematical models from examples—labeled data in supervised learning, unlabeled

In [93]:
def query_with_rag(query):
    print(f"Query: {query}")
    print("-"*100)
    answer = rag_chain_lcel.invoke(query)
    print(f"Answer: {answer}")
    print("-"*100)

    docs = retriever.invoke(query)
    print("Source Documents:")
    for i, doc in enumerate(docs, start=1):
        print(f"Document {i}:")
        print(doc.page_content)
        print("-"*100)

    print("-"*100)

query_with_rag("What is Machine Learning?")

    
    
    
    

Query: What is Machine Learning?
----------------------------------------------------------------------------------------------------
Answer: Machine Learning is a branch of artificial intelligence where systems learn patterns from data without being explicitly programmed with rules for each case. Algorithms create mathematical models from examples like labeled data in supervised learning, unlabeled data in unsupervised learning, or rewards and penalties in reinforcement learning. These models are then used to make predictions or decisions on new inputs. Common applications of Machine Learning include spam filtering and recommendation systems.
----------------------------------------------------------------------------------------------------
Source Documents:
Document 1:
Machine Learning


Machine learning is a branch of artificial intelligence in which systems learn patterns from data instead of being programmed with explicit rules for every case.
Algorithms build mathematical models

# Add New Documents to Existing Vector Store

In [94]:
new_docs = """ 
Reinforcement Learning is a type of machine learning that focuses on training agents to make decisions in an environment by maximizing a reward signal.
It is a type of supervised learning where the agent learns to make decisions by interacting with the environment and receiving feedback in the form of rewards or penalties.Reinforcement learning is used in a wide range of applications, including game playing, robotics, and recommendation systems.
Reinforcement learning differs from other forms of machine learning, such as supervised and unsupervised learning, because the learning process is driven by trial and error. Agents take actions in an environment, observe the outcomes, and use those outcomes (rewards or penalties) to gradually learn an optimal strategy, called a policy, for achieving their goals.
The field has produced robust algorithms like Q-learning and policy gradients, which have been at the core of breakthroughs such as AlphaGo and autonomous robot control. Reinforcement learning is characterized by exploration (trying new actions to discover their effects) and exploitation (using known actions that give high rewards). Balancing these is central to success in reinforcement learning tasks.
Challenges in reinforcement learning often include managing delayed rewards, large or continuous state spaces, and ensuring stability and efficiency when training complex neural networks (deep reinforcement learning).
"""

In [95]:
new_docs

' \nReinforcement Learning is a type of machine learning that focuses on training agents to make decisions in an environment by maximizing a reward signal.\nIt is a type of supervised learning where the agent learns to make decisions by interacting with the environment and receiving feedback in the form of rewards or penalties.Reinforcement learning is used in a wide range of applications, including game playing, robotics, and recommendation systems.\nReinforcement learning differs from other forms of machine learning, such as supervised and unsupervised learning, because the learning process is driven by trial and error. Agents take actions in an environment, observe the outcomes, and use those outcomes (rewards or penalties) to gradually learn an optimal strategy, called a policy, for achieving their goals.\nThe field has produced robust algorithms like Q-learning and policy gradients, which have been at the core of breakthroughs such as AlphaGo and autonomous robot control. Reinforc

In [96]:
new_docs = Document(page_content=new_docs, metadata={"source": "manual_addition", "topic": "Reinforcement Learning"})

In [97]:
vectors_store

In [98]:
new_chunks = text_splitter.split_documents([new_docs])
new_chunks

    

[Document(metadata={'source': 'manual_addition', 'topic': 'Reinforcement Learning'}, page_content='Reinforcement Learning is a type of machine learning that focuses on training agents to make decisions in an environment by maximizing a reward signal.\nIt is a type of supervised learning where the agent learns to make decisions by interacting with the environment and receiving feedback in the form of rewards or penalties.Reinforcement learning is used in a wide range of applications, including game playing, robotics, and recommendation systems.\nReinforcement learning differs from other forms'),
 Document(metadata={'source': 'manual_addition', 'topic': 'Reinforcement Learning'}, page_content='learning differs from other forms of machine learning, such as supervised and unsupervised learning, because the learning process is driven by trial and error. Agents take actions in an environment, observe the outcomes, and use those outcomes (rewards or penalties) to gradually learn an optimal st

In [99]:
vectors_store.add_documents(new_chunks)

['fef97783-a6df-4048-8957-09547a3ab73e',
 '47065d85-5b74-4c8f-bbd2-57f2c2288dee',
 '49356681-993e-40cd-88ea-a1f311f02451',
 'bd402b2d-168c-4873-bcb7-b0d5831f11a0']

In [100]:
vectors_store._collection.count()

26

In [101]:
new_query = "What is Reinforcement Learning?"
result = query_with_rag(new_query)
result








Query: What is Reinforcement Learning?
----------------------------------------------------------------------------------------------------
Answer: Reinforcement Learning is a type of machine learning that focuses on training agents to make decisions in an environment by maximizing a reward signal. It involves the agent interacting with the environment, receiving feedback in the form of rewards or penalties, and adjusting its actions to achieve the highest possible reward. Reinforcement learning is used in various applications, such as game playing, robotics, and recommendation systems, and is characterized by exploration and exploitation to find the most optimal actions.
----------------------------------------------------------------------------------------------------
Source Documents:
Document 1:
Reinforcement Learning is a type of machine learning that focuses on training agents to make decisions in an environment by maximizing a reward signal.
It is a type of supervised learning 

# Advanced Rag Techniques - Conversational Memory

In [102]:
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, AIMessage



In [103]:
from langchain_core.prompts import MessagesPlaceholder


contextualize_q_system_prompt = """
Given a chat history and the latest user question which might reference context in the chat history, formulate a standalone question which can be understood without the chat history.
Do not answer the question, only reformulate it if needed and otherwise return it as is.
"""

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

In [104]:
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_q_prompt
)
history_aware_retriever

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x13778c410>, search_kwargs={'k': 3}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk

In [107]:
qa_system_prompt = """ You are an assistant for questions answering tasks. Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know. Don't make up an answer.
Use three maximum sentences to answer the question.
Context: {context}
"""

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=qa_prompt,
)

ConversationChain = create_retrieval_chain(
    retriever=history_aware_retriever,
    combine_docs_chain=question_answer_chain,
)



In [109]:
chat_history = []

result1 = ConversationChain.invoke({"input": "What is Machine Learning?", "chat_history": chat_history})

print(f"What is Machine Learning?")
print("-"*100)
print(f"Answer:{result1['answer']}")
print("-"*100)







What is Machine Learning?
----------------------------------------------------------------------------------------------------
Answer:Machine learning is a branch of artificial intelligence where systems learn patterns from data instead of being programmed with specific rules for each scenario. Algorithms create mathematical models using labeled or unlabeled data in supervised or unsupervised learning, or rewards and penalties in reinforcement learning, to make predictions or decisions on new inputs. Common applications of machine learning include spam filtering and recommendation systems.
----------------------------------------------------------------------------------------------------


In [110]:
chat_history.extend(
    [HumanMessage(content="What is Machine Learning?"),
     AIMessage(content=result1["answer"])]
)

result2 = ConversationChain.invoke({"input": "What are its applications?", "chat_history": chat_history})

print(f"What are its applications?")
print("-"*100)
print(f"Answer:{result2['answer']}")
print("-"*100)





What are its applications?
----------------------------------------------------------------------------------------------------
Answer:Common applications of machine learning include spam filtering, recommendation systems, image and speech recognition, medical diagnosis, financial fraud detection, and autonomous vehicles. Machine learning is also used in predicting customer behavior, stock market trends, and optimizing supply chain management. It has a wide range of applications across various industries and sectors.
----------------------------------------------------------------------------------------------------


In [111]:
result2

{'input': 'What are its applications?',
 'chat_history': [HumanMessage(content='What is Machine Learning?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Machine learning is a branch of artificial intelligence where systems learn patterns from data instead of being programmed with specific rules for each scenario. Algorithms create mathematical models using labeled or unlabeled data in supervised or unsupervised learning, or rewards and penalties in reinforcement learning, to make predictions or decisions on new inputs. Common applications of machine learning include spam filtering and recommendation systems.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
 'context': [Document(id='c83a6ed1-fcfb-46e3-b6b9-111b2a00e7c0', metadata={'source': 'data/doc_0.txt'}, page_content='Machine Learning\n\n\nMachine learning is a branch of artificial intelligence in which systems learn patterns from data instead of being programmed with explic

In [112]:
llm 

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11', 'langchain-openai': '1.3.3'}}, output_version=None, profile={'name': 'GPT-3.5-turbo', 'release_date': '2023-03-01', 'last_updated': '2023-11-06', 'open_weights': False, 'max_input_tokens': 16385, 'max_output_tokens': 4096, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': False, 'structured_output': False, 'attachment': False, 'temperature': True, 'image_url_inputs': False, 'pdf_inputs': False, 'pdf_tool_message': False, 'image_tool_message': False, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x1377957f0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x137569d30>, root_client=<openai.OpenAI object at 0x1